In [7]:
#imports 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import f_classif

In [8]:
df=pd.read_csv("telecom_cus.csv")
print(df.index)
print("All columns are")
ai=0
for i in df:
    print(ai,i)
    ai+=1

RangeIndex(start=0, stop=1000, step=1)
All columns are
0 region
1 tenure
2 age
3 marital
4 address
5 income
6 ed
7 employ
8 retire
9 gender
10 reside
11 custcat


In [9]:
# As the data has already been encoded now forst let's test without scaling
tx = df.iloc[:,:11]
ty = df.iloc[:,11]
print(tx)
print(ty)

     region  tenure  age  marital  address  income  ed  employ  retire  \
0         2      13   44        1        9      64   4       5       0   
1         3      11   33        1        7     136   5       5       0   
2         3      68   52        1       24     116   1      29       0   
3         2      33   33        0       12      33   2       0       0   
4         2      23   30        1        9      30   1       2       0   
..      ...     ...  ...      ...      ...     ...  ..     ...     ...   
995       3      10   39        0        0      27   3       0       0   
996       1       7   34        0        2      22   5       5       0   
997       3      67   59        0       40     944   5      33       0   
998       3      70   49        0       18      87   2      22       0   
999       3      50   36        1        7      39   3       3       0   

     gender  reside  
0         0       2  
1         0       6  
2         1       2  
3         1       1  
4

In [10]:
tX_train, tX_test, ty_train, ty_test = train_test_split( tx, ty,test_size=0.2)

tmodel = LogisticRegression(max_iter=5000)

tmodel.fit(tX_train, ty_train)


ty_pred = tmodel.predict(tX_test)


print("Accuracy:", accuracy_score(ty_test, ty_pred))

print("\nClassification Report:")
print(classification_report(ty_test, ty_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(ty_test, ty_pred))

Accuracy: 0.41

Classification Report:
              precision    recall  f1-score   support

           1       0.48      0.50      0.49        58
           2       0.25      0.24      0.25        37
           3       0.45      0.45      0.45        58
           4       0.39      0.38      0.39        47

    accuracy                           0.41       200
   macro avg       0.39      0.39      0.39       200
weighted avg       0.41      0.41      0.41       200


Confusion Matrix:
[[29  5 18  6]
 [ 6  9  9 13]
 [11 12 26  9]
 [14 10  5 18]]


In [11]:
tscores = cross_val_score(tmodel, tx, ty, cv=10, scoring="accuracy")

print('Actual cross-val results')
print(f"Scores for each fold: {tscores}")
print(f"Mean Accuracy: {tscores.mean():.2f}")
print(f"Standard Deviation: {tscores.std():.2f}")

Actual cross-val results
Scores for each fold: [0.42 0.51 0.3  0.35 0.4  0.36 0.42 0.42 0.44 0.51]
Mean Accuracy: 0.41
Standard Deviation: 0.06


In [12]:
print('''That was a pretty terrible score 
Now let's scale the data and drop useless columns
''')

That was a pretty terrible score 
Now let's scale the data and drop useless columns



In [13]:
fx = df.drop(columns=["custcat"])
fy = df["custcat"]

In [14]:
fscores, pvalues = f_classif(fx, fy)

anovadf = pd.DataFrame({"Feature": fx.columns,"F-Score": fscores,"p-value": pvalues})

anovadf = anovadf.sort_values(by="F-Score",ascending=False)

print(anovadf.round(4))

    Feature  F-Score  p-value
6        ed  61.4543   0.0000
1    tenure  41.3101   0.0000
7    employ  16.9757   0.0000
4   address   8.4329   0.0000
2       age   7.5214   0.0001
5    income   6.6894   0.0002
10   reside   3.9765   0.0079
3   marital   3.4995   0.0151
8    retire   3.0047   0.0296
0    region   1.0949   0.3503
9    gender   0.3730   0.7725


In [15]:
print('We can safely drop region and gender')

We can safely drop region and gender


In [16]:
df=pd.read_csv("telecom_cus.csv")
print(df.index)
print("All columns are")
ai=0
for i in df:
    print(ai,i)
    ai+=1

RangeIndex(start=0, stop=1000, step=1)
All columns are
0 region
1 tenure
2 age
3 marital
4 address
5 income
6 ed
7 employ
8 retire
9 gender
10 reside
11 custcat


In [17]:
aadf = df.drop(['region','gender'],axis=1)
print(aadf.columns)

Index(['tenure', 'age', 'marital', 'address', 'income', 'ed', 'employ',
       'retire', 'reside', 'custcat'],
      dtype='object')


In [18]:
X = aadf.drop('custcat',axis=1)
y = df['custcat']
print(X.columns)
print(y)

Index(['tenure', 'age', 'marital', 'address', 'income', 'ed', 'employ',
       'retire', 'reside'],
      dtype='object')
0      1
1      4
2      3
3      1
4      3
      ..
995    1
996    1
997    4
998    3
999    2
Name: custcat, Length: 1000, dtype: int64


In [23]:
# First let's test using standard scaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,)


preprocessor = ColumnTransformer([
    ('standard', StandardScaler(), ['tenure', 'age', 'marital', 'address', 'income', 'ed', 'employ','retire', 'reside']),
], remainder='passthrough')


pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

In [24]:
X_train_transformed = pipeline.named_steps['preprocessor'].transform(X_train)

feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

transformed_df = pd.DataFrame(
    X_train_transformed,
    columns=feature_names
)

print(transformed_df.head())

print()

pt = pipeline.named_steps['preprocessor'].named_transformers_['standard']


   standard__tenure  standard__age  standard__marital  standard__address  \
0          0.692701       0.811411           0.997503           1.046839   
1          0.879728       1.359595          -1.002503           1.639669   
2         -0.102164      -0.128334           0.997503           0.256399   
3          0.412161      -0.363270           0.997503          -1.028066   
4         -0.429461      -0.206646          -1.002503          -0.336431   

   standard__income  standard__ed  standard__employ  standard__retire  \
0          0.723723     -0.587830          1.572543         -0.229416   
1         -0.443888     -0.587830          0.393136         -0.229416   
2         -0.522310      0.235750         -0.589704         -0.229416   
3         -0.330613      1.059329         -0.294852         -0.229416   
4          0.227053      0.235750          0.196568         -0.229416   

   standard__reside  
0         -0.238574  
1         -0.927597  
2         -0.238574  
3          2.517

In [25]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")

print(classification_report(y_test, y_pred))

Accuracy: 0.43

Classification Report:
              precision    recall  f1-score   support

           1       0.41      0.52      0.46        52
           2       0.31      0.25      0.28        40
           3       0.42      0.46      0.44        59
           4       0.58      0.45      0.51        49

    accuracy                           0.43       200
   macro avg       0.43      0.42      0.42       200
weighted avg       0.44      0.43      0.43       200



In [26]:
# Checking through pipeline
scores = cross_val_score(pipeline ,X ,y ,cv=10,scoring='accuracy')

print("Cross Validation Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())

Cross Validation Scores: [0.39 0.5  0.32 0.36 0.41 0.33 0.41 0.45 0.42 0.47]
Mean Accuracy: 0.40599999999999997
Standard Deviation: 0.05535341001239218


In [27]:
print('Not much improvement but now it takes less number of iterations')

Not much improvement but now it takes less number of iterations


In [38]:
# Now let's check using Yeo-Johnson

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,)


preprocessor = ColumnTransformer([
    ('yeo_johnson', PowerTransformer(method='yeo-johnson', standardize=True), ['tenure', 'age', 'marital', 'address', 'income',
                                                                               'ed', 'employ','retire', 'reside']),
], remainder='passthrough')


pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='lbfgs'))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

In [39]:
X_train_transformed = pipeline.named_steps['preprocessor'].transform(X_train)

feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

transformed_df = pd.DataFrame(
    X_train_transformed,
    columns=feature_names
)

print(transformed_df.head())

print()

pt = pipeline.named_steps['preprocessor'].named_transformers_['yeo_johnson']

print(pt.lambdas_)

   yeo_johnson__tenure  yeo_johnson__age  yeo_johnson__marital  \
0            -0.221965         -1.034236             -0.997503   
1             0.602618         -0.563731              1.002503   
2            -1.232213          0.103468              1.002503   
3             1.483996          1.933859             -0.997503   
4             1.518104          1.706240              1.002503   

   yeo_johnson__address  yeo_johnson__income  yeo_johnson__ed  \
0             -0.651158            -1.093670         0.343387   
1             -0.651158             0.349441        -0.483911   
2              0.156311             1.857191         1.069531   
3              1.477637             1.305287         0.343387   
4              2.003620             0.835949        -0.483911   

   yeo_johnson__employ  yeo_johnson__retire  yeo_johnson__reside  
0            -0.359669            -0.223313             0.079335  
1             0.507059            -0.223313             1.213271  
2          

In [40]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.435

Classification Report:
              precision    recall  f1-score   support

           1       0.47      0.54      0.50        52
           2       0.41      0.32      0.36        41
           3       0.43      0.48      0.45        56
           4       0.42      0.37      0.40        51

    accuracy                           0.43       200
   macro avg       0.43      0.43      0.43       200
weighted avg       0.43      0.43      0.43       200


Confusion Matrix:
[[28  2 14  8]
 [ 4 13 14 10]
 [13  8 27  8]
 [15  9  8 19]]


In [41]:
# Checking through pipeline
scores = cross_val_score(pipeline ,X ,y ,cv=10,scoring='accuracy')

print("Cross Validation Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())

Cross Validation Scores: [0.45 0.5  0.3  0.39 0.4  0.35 0.43 0.44 0.43 0.44]
Mean Accuracy: 0.413
Standard Deviation: 0.053301031884945727


In [32]:
fscores, pvalues = f_classif(X, y)

anovadf = pd.DataFrame({"Feature": X.columns,"F-Score": fscores,"p-value": pvalues})

anovadf = anovadf.sort_values(by="F-Score",ascending=False)

print(anovadf.round(4))

   Feature  F-Score  p-value
5       ed  61.4543   0.0000
0   tenure  41.3101   0.0000
6   employ  16.9757   0.0000
3  address   8.4329   0.0000
1      age   7.5214   0.0001
4   income   6.6894   0.0002
8   reside   3.9765   0.0079
2  marital   3.4995   0.0151
7   retire   3.0047   0.0296


In [33]:
# Lets check after dropping more columns dropping (reside,marital,retire)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,)


preprocessor = ColumnTransformer([
    ('yeo_johnson', PowerTransformer(method='yeo-johnson', standardize=True), ['tenure', 'age', 'address',
                                                                               'income','ed', 'employ']),
], remainder='passthrough')


pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='lbfgs'))
])

pipeline.fit(X_train, y_train)



# Checking through pipeline
scores = cross_val_score(pipeline ,X ,y ,cv=10,scoring='accuracy')

print("Cross Validation Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())

Cross Validation Scores: [0.45 0.51 0.28 0.4  0.41 0.33 0.39 0.41 0.41 0.47]
Mean Accuracy: 0.406
Standard Deviation: 0.062321745803531524


In [34]:
# Lets check after dropping more columns dropping (reside,marital,retire,income,age,address)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,)


preprocessor = ColumnTransformer([
    ('yeo_johnson', PowerTransformer(method='yeo-johnson', standardize=True), ['tenure','ed', 'employ']),
], remainder='passthrough')


pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='lbfgs',max_iter=5000))
])

pipeline.fit(X_train, y_train)



# Checking through pipeline
scores = cross_val_score(pipeline ,X ,y ,cv=10,scoring='accuracy')

print("Cross Validation Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())

print(pipeline)

Cross Validation Scores: [0.41 0.5  0.33 0.39 0.42 0.31 0.4  0.44 0.4  0.46]
Mean Accuracy: 0.406
Standard Deviation: 0.053329166503893535
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('yeo_johnson',
                                                  PowerTransformer(),
                                                  ['tenure', 'ed',
                                                   'employ'])])),
                ('classifier', LogisticRegression(max_iter=5000))])
